In [1]:
import pandas as pd
import category_encoders as ce
import numpy as np

np.random.seed(42)

# 1. Treatment Coding (Dummy Coding)

In [2]:
from patsy.contrasts import Treatment
levels = ['X-Small', 'Small', 'Medium', 'Large', 'X-Large']
contrast = Treatment(reference='Medium').code_without_intercept(levels)
print(contrast.matrix)
print(contrast.column_suffixes)

[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
['[T.X-Small]', '[T.Small]', '[T.Large]', '[T.X-Large]']


In [3]:
from patsy.contrasts import Treatment
levels = ['X-Small', 'Small', 'Medium', 'Large', 'X-Large']
contrast = Treatment(reference='Medium').code_with_intercept(levels)
print(contrast.matrix)
print(contrast.column_suffixes)

[[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]]
['[X-Small]', '[Small]', '[Medium]', '[Large]', '[X-Large]']


In [4]:
df = pd.DataFrame({
	'Sector': ['Tech', 'Health', 'Finance', 'Energy', 'Tech',
	'Health', 'Finance', 'Energy'] * 3,
	'Return': [12, 15, 8, 5, 13, 14, 9, 6, 11, 16, 7, 4, 14, 15, 10, 5, 12, 17, 8, 5, 13, 16, 9, 6]
})

In [5]:
levels = ['Tech', 'Health', 'Finance', 'Energy']
contrast = Treatment(reference='Finance').code_without_intercept(levels)

In [6]:
print(contrast.matrix)
print(contrast.column_suffixes)

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 0.]
 [0. 0. 1.]]
['[T.Tech]', '[T.Health]', '[T.Energy]']


# 2. Sum Coding (Deviation / Effect Coding)

### 1. The Python Matrix Generation (Example 1)

In [7]:
from category_encoders import SumEncoder

# Create sample data: Stock returns by sector
df = pd.DataFrame({
	'Sector': ['Tech', 'Health', 'Finance', 'Energy', 'Tech',
	'Health', 'Finance', 'Energy'] * 3,
	'Return': [12, 15, 8, 5, 13, 14, 9, 6, 11, 16, 7, 4, 14, 15, 10, 5, 12, 17, 8, 5, 13, 16, 9, 6]
})

In [8]:
# Sum (Effect) Coding
encoder_sum = ce.SumEncoder(cols=['Sector'])
df_sum = encoder_sum.fit_transform(df)
print("\n2. SUM CODING (Compares to grand mean):")
print(df_sum.head())


2. SUM CODING (Compares to grand mean):
   Sector_0  Sector_1  Sector_2  Return
0       1.0       0.0       0.0      12
1       0.0       1.0       0.0      15
2       0.0       0.0       1.0       8
3      -1.0      -1.0      -1.0       5
4       1.0       0.0       0.0      13


How to Read the Matrix

In a Sum Coding matrix of $N$ categories, you still get $N-1$ columns. However, the dropped category is not represented by all zeros; it is represented by **-1s**.

- **Row 1 (`Tech`): `[1. 0.]`**
    This turns on the `[S.Tech]` column. Its coefficient will equal the Tech sector's exact deviation from the grand mean.
- **Row 2 (`Health`): `[0. 1.]`**
    This turns on the `[S.Health]` column. Its coefficient will equal the Health sector's exact deviation from the grand mean.
- **Row 3 (`Finance`): `[-1. -1.]`**

Finance is the dropped "reference" category. Because its row is filled with `-1`s, the model does not generate an explicit `[S.Finance]` coefficient.

### 3. The Financial Example: Interpreting the Results

Imagine you run a regression to predict **Annual Portfolio Return (%)** based purely on these three sectors.
If you used **Sum Coding**, your regression output would look something like this:

- **Intercept:** $10.0$
- **`[S.Tech]` Coefficient:** $+2.5$
- **`[S.Health]` Coefficient:** $-1.0$

**Here is how you interpret these numbers as an analyst:**

1. **The Intercept is the Grand Mean:** The average return across _all three sectors combined_ is $10.0\%$. (Note: In Treatment coding, the intercept would have been just the Tech average).
2. **The Tech Effect:** The Tech sector returns $2.5\%$ _above_ the market average. (Actual Tech average = $12.5\%$).
3. **The Health Effect:** The Health sector returns $1.0\%$ _below_ the market average. (Actual Health average = $9.0\%$).

### 4. How to find the "Missing" Finance Category

Because the Finance row in our matrix was `[-1. -1.]`, the mathematical rule of Sum Coding is that **the sum of all category effects must equal zero**.

To find the hidden coefficient for Finance, you simply calculate the negative sum of the other coefficients:
$$
\begin{align}
\text{Finance Effect} &= - ( \text{Tech Effect} + \text{Health Effect} ) \\ 
\text{Finance Effect} &= - ( 2.5 + (-1.0) ) \\
\text{Finance Effect} &= - 1.5
\end{align}
$$
**The Finance Effect:** The Finance sector returns $1.5\%$ _below_ the market average. (Actual Finance average = $8.5\%$).

---

### Example 2

In [9]:
from sklearn.datasets import fetch_openml
bunch = fetch_openml(name='house_prices', as_frame=True)
display_cols = [
    'Id',
    'MSSubClass',
    'MSZoning',
    'LotFrontage',
    'YearBuilt',
    'Heating',
    'CentralAir',
]
y = bunch.target
X = pd.DataFrame(bunch.data, columns=bunch.feature_names)[display_cols]
enc = SumEncoder(cols=['CentralAir', 'Heating']).fit(X, y)
numeric_dataset = enc.transform(X)
print(numeric_dataset.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Id            1460 non-null   int64  
 1   MSSubClass    1460 non-null   int64  
 2   MSZoning      1460 non-null   object 
 3   LotFrontage   1201 non-null   float64
 4   YearBuilt     1460 non-null   int64  
 5   Heating_0     1460 non-null   float64
 6   Heating_1     1460 non-null   float64
 7   Heating_2     1460 non-null   float64
 8   Heating_3     1460 non-null   float64
 9   Heating_4     1460 non-null   float64
 10  CentralAir_0  1460 non-null   float64
dtypes: float64(7), int64(3), object(1)
memory usage: 125.6+ KB
None


# 3. Backward Difference Coding

In [10]:
df_ordinal = pd.DataFrame({
    'Rating': ['B', 'BB', 'BBB', 'A', 'B', 'BB', 'BBB', 'A'] * 3,
    'DefaultRate': [8.5, 5.2, 2.1, 0.8, 9.0, 5.8, 2.3, 0.9,
                    8.2, 5.0, 2.0, 0.7, 8.8, 5.5, 2.2, 0.8,
                    8.6, 5.3, 2.1, 0.8, 8.9, 5.6, 2.4, 0.9]
})

### Step 1: Calculating the Grouped Means

In [11]:
print("\nOrdinal Data (Credit Ratings):")
print(df_ordinal.groupby('Rating')['DefaultRate'].mean())


Ordinal Data (Credit Ratings):
Rating
A      0.816667
B      8.666667
BB     5.400000
BBB    2.183333
Name: DefaultRate, dtype: float64


### Step 2: Defining the Ordinal Order

**What it does?**
Backward Difference Coding only works if the model knows the exact "ladder" of your data. If you didn't do this, the encoder would use the alphabetical sort and calculate the difference between 'A' and 'B', which makes no financial sense. This step forces the data into a strict hierarchical ladder.

In [12]:
# Define the correct order
rating_order = ['B', 'BB', 'BBB', 'A']
df_ordinal['Rating'] = pd.Categorical(df_ordinal['Rating'], categories=rating_order, ordered=True)

### Step 3: The Backward Difference Matrix Generation
**What it does?**
- It takes your 4 categories ($N=4$) and creates $N-1$ columns (3 columns). 
- The goal of these specific fractions is to create an orthogonal matrix so that when you run a linear regression, the resulting coefficients represent the exact difference between the mean of one level and the mean of the previous level.
- The Mathematical Formula for the Fractions:
    - Let $N$ equal the total number of categories ($4$).
    - Let $j$ equal the index of the new column being created (0, 1, or 2).
    - Let $i$ equal the index of the category row (B=0, BB=1, BBB=2, A=3).

**For any given cell in the matrix**:
- If the row's category is below or equal to the comparison threshold ($i \le j$), the formula is:$$- \frac{N - (j + 1)}{N}$$
- If the row's category is above the comparison threshold ($i > j$), the formula is:$$\frac{j + 1}{N}$$

In [13]:
# Backward Difference Coding
encoder_backward = ce.BackwardDifferenceEncoder(cols=['Rating'])
df_backward = encoder_backward.fit_transform(df_ordinal)

**Calculation Breakdown by Column:1**. 
1. Column Rating_0 ($j = 0$): Compares BB vs. B
    - Row B ($i=0$): $0 \le 0$, so $- \frac{4 - 1}{4} = \mathbf{-0.75}$
    - Row BB ($i=1$): $1 > 0$, so $\frac{0 + 1}{4} = \mathbf{0.25}$
    - Row BBB ($i=2$): $2 > 0$, so $\frac{0 + 1}{4} = \mathbf{0.25}$
    - Row A ($i=3$): $3 > 0$, so $\frac{0 + 1}{4} = \mathbf{0.25}$

2. Column Rating_1 ($j = 1$): Compares BBB vs. BB
    - Row B ($i=0$): $0 \le 1$, so $- \frac{4 - 2}{4} = \mathbf{-0.50}$
    - Row BB ($i=1$): $1 \le 1$, so $- \frac{4 - 2}{4} = \mathbf{-0.50}$
    - Row BBB ($i=2$): $2 > 1$, so $\frac{1 + 1}{4} = \mathbf{0.50}$
    - Row A ($i=3$): $3 > 1$, so $\frac{1 + 1}{4} = \mathbf{0.50}$

3. Column Rating_2 ($j = 2$): Compares A vs. BBB
    - Row B ($i=0$): $0 \le 2$, so $- \frac{4 - 3}{4} = \mathbf{-0.25}$
    - Row BB ($i=1$): $1 \le 2$, so $- \frac{4 - 3}{4} = \mathbf{-0.25}$
    - Row BBB ($i=2$): $2 \le 2$, so $- \frac{4 - 3}{4} = \mathbf{-0.25}$
    - Row A ($i=3$): $3 > 2$, so $\frac{2 + 1}{4} = \mathbf{0.75}$

In [14]:
print("\nBACKWARD DIFFERENCE CODING:")
print(df_backward.head())
print("\nInterpretation: Compares BB vs B, BBB vs BB, A vs BBB")


BACKWARD DIFFERENCE CODING:
   Rating_0  Rating_1  Rating_2  DefaultRate
0     -0.75      -0.5     -0.25          8.5
1      0.25      -0.5     -0.25          5.2
2      0.25       0.5     -0.25          2.1
3      0.25       0.5      0.75          0.8
4     -0.75      -0.5     -0.25          9.0

Interpretation: Compares BB vs B, BBB vs BB, A vs BBB


**How to Read the Matrix?**
- Unlike Dummy or Sum coding, which use simple 1s, 0s, and -1s, the Backward Difference matrix uses fractions. 
- This fractional weighting is a mathematical algebraic trick (creating an orthogonal matrix) designed to force the regression model to output the exact step-by-step differences.

You only need to understand what the columns represent:
- [Rating_0] Column: Represents the shift from B to BB.
- [Rating_1] Column: Represents the shift from BB to BBB.
- [Rating_2] Column: Represents the shift from BBB to A.

---

### The Financial Example: Interpreting the Results
Imagine you run a regression to predict the Interest Rate Yield (%) a company must pay on its corporate bonds, based purely on its Credit Rating.

If you used Backward Difference Coding, your regression output would look like this:
- Intercept: 6.0
- [Rating_0] Coefficient: -1.5
- [Rating_1] Coefficient: -1.0
- [Rating_2] Coefficient: -0.5

Here is how you interpret these numbers as an analyst:
- The Intercept is the Grand Mean: The average bond yield across all ratings in your dataset is 6.0%.
- **The BB vs. B Effect ([Rating_0])**: Upgrading a company's rating from **B to BB** lowers their interest rate yield by exactly 1.5%.
- **The BBB vs. BB Effect ([Rating_1])**: Upgrading from **BB to BBB** lowers their yield by an additional 1.0%.
- **The A vs. BBB Effect ([Rating_2])**: Upgrading from **BBB to A** lowers their yield by an additional 0.5%.

The fractions simply trick the regression math into calculating the exact step-up value between your credit tiers.